# ORTHRUS-MSTC-PIDS — PAI-DSW AllInOne Notebook

本 Notebook 是 **ORTHRUS-MSTC-PIDS** 项目在阿里云 PAI-DSW 上的 AllInOne
入口。它与既有 Google Colab 版本
`notebooks/ORTHRUS_MSTC_PIDS_AllInOne_Colab.ipynb` 语义一致，但将：

* Google Drive → `/mnt/workspace/mstc_pids`
* Colab clone → 本地 GitHub ZIP 安全解压
* Colab `/content/orthrus` → `/mnt/workspace/.../<repo-top>`
* PostgreSQL 自动安装 → 已存在的 artifacts 直接复用 detection_only

> **重要使用约定**
>
> 1. 数据目录已经在 PAI-DSW 中准备：
>    `/mnt/workspace/mstc_pids`（即原 Google Drive `mstc_pids` 的完整副本）。
> 2. 源码以 GitHub branch ZIP 形式上传：
>    `/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist.zip`。
> 3. **所有重任务默认都是关闭的**（preprocessing / smoke / main matrix /
>    ablations / resume）。
> 4. **不要直接 Run All 后离开**。先按顺序运行 0–9 单元格完成 setup，
>    再按需打开 smoke → main matrix。
> 5. Smoke 通过后再设置 `RUN_MAIN_MATRIX=True`，然后运行 Main Matrix 单元格。
> 6. 当 build_graphs / metadata / embed_nodes / embed_edges 全部完整时，
>    完全不需要 PostgreSQL，pipeline 自动以 `detection_only` 模式运行。
> 7. 本 Notebook **不会**删除或破坏 `/mnt/workspace` 下任何已有文件；
>    解压源码时使用 staging 目录，验证成功后才会切换。
> 8. 既有 Google Colab Notebook 仍然保留，本 Notebook 是独立新增文件，
>    不修改、不替换原 Colab Notebook。


PAI Notebook 同时支持两种源码使用方式：
1. `/mnt/workspace` 下保留原 GitHub ZIP；
2. ZIP 已提前解压、原 ZIP 已删除的预解压源码目录。

当预解压源码通过 required-file validation 时，默认优先复用，不重复解压。


## 0. Unified Parameters

In [ ]:
from pathlib import Path

WORKSPACE_ROOT = Path("/mnt/workspace")
MSTC_ROOT = WORKSPACE_ROOT / "mstc_pids"
ARTIFACT_ROOT = MSTC_ROOT / "artifacts"
DATA_ROOT = MSTC_ROOT / "data"

# -------------------------------------------------------------------------
# PRODUCTION SOURCE PARAMETERS (fix/c8-main-matrix-runtime-failures)
# Active production source must resolve to exact GitHub commit 08ee62a.
# -------------------------------------------------------------------------
PRODUCTION_SOURCE_COMMIT = (
    "08ee62ac86d2c7e70f2bf49a18fd888e9227c448"
)

REPOSITORY_REF = "fix/c8-main-matrix-runtime-failures"

# Primary: exact-commit pre-extracted source directory.
PREEXTRACTED_PROJECT_ROOT = Path(
    "/mnt/workspace/"
    "orthrus-08ee62ac86d2c7e70f2bf49a18fd888e9227c448"
)
PREFER_PREEXTRACTED_SOURCE = True

# Primary: exact-commit ZIP archive.
SOURCE_ARCHIVE = (
    WORKSPACE_ROOT /
    f"orthrus-{PRODUCTION_SOURCE_COMMIT}.zip"
)
SOURCE_KIND = "github_zip"

# Legacy fallback paths (NOT active production source):
# - Ground Truth persistent fallback
# - Old config absolute paths (for config-id preservation)
LEGACY_SOURCE_ARCHIVE = (
    WORKSPACE_ROOT / "orthrus-fix-c8-loader-telemetry-persist.zip"
)
LEGACY_PREEXTRACTED_PROJECT_ROOT = Path(
    "/mnt/workspace/"
    "orthrus-fix-c8-loader-telemetry-persist/"
    "orthrus-fix-c8-loader-telemetry-persist"
)
LEGACY_REPOSITORY_REF = "fix/c8-loader-telemetry-persist"

# Expected production commit (for cross-check only, now superseded by
# PRODUCTION_SOURCE_COMMIT + source provenance verification).
EXPECTED_PRODUCTION_COMMIT = PRODUCTION_SOURCE_COMMIT

DATASET = "THEIA_E3"
SEEDS = [0]
CORPUS_SCOPE = "train_only"

# ----- Source preparation / extraction --------------------------------
EXTRACT_SOURCE = True
FORCE_EXTRACT_SOURCE = False

# ----- Dependencies ---------------------------------------------------
INSTALL_DEPENDENCIES = True

# ----- Database ------------------------------------------------------
ALLOW_NETWORK_FETCH_GROUND_TRUTH = True
RUN_DB_PREFLIGHT = False
ORTHRUS_DB_HOST = ""
ORTHRUS_DB_PORT = ""
ORTHRUS_DB_USER = ""
ORTHRUS_DB_PASSWORD = ""

# ----- Preprocessing -------------------------------------------------
RUN_PREPROCESS_BENCHMARK = False
RUN_PREPROCESS = False
FORCE_PREPROCESS = False
PREPROCESS_SUBSTAGES = (
    "build_graphs,embed_nodes,embed_edges"
)

# ----- Smoke / Main / Ablations / Resume ------------------------------
RUN_BASELINE_SMOKE = False
SMOKE_MAX_WINDOWS_PER_SPLIT = 2

RUN_MAIN_MATRIX = False
FORCE_MAIN_MATRIX = False

RUN_ABLATIONS = False
FORCE_ABLATIONS = False

RUN_MANUAL_RESUME = False
EXPERIMENT_GROUP = "ablation"
RESUME_CONFIG_REL = "config/experiments/mstc_full.yml"
CHECKPOINT = Path("")

# ----- Recovery -------------------------------------------------------
# Recovery mode must be explicitly enabled by user.
RECOVERY_MAIN_MATRIX = False

# ----- Result collection / display -----------------------------------
RUN_COLLECT_EXPORT = False
RUN_DISPLAY_RESULTS = False

assert DATASET in {"THEIA_E3", "THEIA_E5"}
assert CORPUS_SCOPE in {"train_only", "official_full_dataset"}

print("=" * 60)
print("Unified Parameters")
print("=" * 60)
print(f"Dataset                   = {DATASET}")
print(f"corpus_scope              = {CORPUS_SCOPE}")
print(f"REPOSITORY_REF            = {REPOSITORY_REF}")
print(f"PRODUCTION_SOURCE_COMMIT  = {PRODUCTION_SOURCE_COMMIT}")
print(f"SOURCE_ARCHIVE            = {SOURCE_ARCHIVE}")
print(f"PREEXTRACTED_PROJECT_ROOT = {PREEXTRACTED_PROJECT_ROOT}")
print(f"LEGACY_SOURCE_ARCHIVE     = {LEGACY_SOURCE_ARCHIVE}")
print(f"LEGACY_PREEXTRACTED_ROOT  = {LEGACY_PREEXTRACTED_PROJECT_ROOT}")
print("All heavy-stage switches are disabled by default.")
print("RECOVERY_MAIN_MATRIX is False; user must explicitly enable.")
print("=" * 60)
PAI_MSTC_ROOT = "/mnt/workspace/mstc_pids"
PAI_SOURCE_ARCHIVE = str(SOURCE_ARCHIVE)
assert str(MSTC_ROOT) == PAI_MSTC_ROOT
assert str(SOURCE_ARCHIVE) == PAI_SOURCE_ARCHIVE


In [ ]:
import os

# Runtime workspace initialization
# This cell MUST run before any cell that uses ENVIRONMENT_DIR

ENVIRONMENT_DIR = ARTIFACT_ROOT / "environment"
ENVIRONMENT_DIR.mkdir(parents=True, exist_ok=True)

# Ensure all workspace roots exist
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
MSTC_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Set environment variables for subprocess access
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)

print("Workspace roots initialized:")
print(f"  Workspace root:   {WORKSPACE_ROOT}")
print(f"  MSTC root:        {MSTC_ROOT}")
print(f"  Artifact root:    {ARTIFACT_ROOT}")
print(f"  Data root:        {DATA_ROOT}")
print(f"  Environment dir:   {ENVIRONMENT_DIR}")


## 1. PAI-DSW Workspace and Source Preparation

In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path


def sha256_file(path: Path) -> str:
    """Streamed SHA-256 for the source archive."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


REQUIRED_SOURCE_PATHS = [
    "src/experiments/run_experiment.py",
    "src/experiments/run_matrix.py",
    "src/run_metadata.py",
    "src/config.py",
    "config/orthrus.yml",
    "config/experiments/baseline.yml",
    "config/experiments/mstc_full.yml",
]


def verify_project_root(root: Path) -> None:
    """Hard-fail when the extracted tree is missing expected files."""
    missing = [p for p in REQUIRED_SOURCE_PATHS
               if not (root / p).is_file()]
    if missing:
        raise FileNotFoundError(
            "Source tree missing required files: " + ", ".join(missing)
        )


def _detect_zip_top_dir(archive: Path) -> str:
    """Pick the single top-level directory inside the ZIP.

    GitHub ZIP archives always wrap everything in one folder.
    Reject archives with multiple distinct top-level entries or
    with loose files at the root, so we never silently flatten the
    user-supplied archive.
    """
    with zipfile.ZipFile(archive, "r") as zf:
        names = zf.namelist()
    tops = sorted({Path(n).parts[0] for n in names if n.strip()})
    if len(tops) != 1:
        raise RuntimeError(
            "Source archive is expected to wrap a single top-level "
            "directory; got: " + ", ".join(tops)
        )
    return tops[0]


def prepare_source_archive(
    archive: Path,
    workspace: Path,
    *,
    force: bool = False,
) -> Path:
    """Extract the GitHub ZIP into a per-build directory.

    - Missing archive -> FileNotFoundError.
    - Existing valid source tree + force=False -> reuse.
    - force=True -> extract into a staging dir, validate, then move.
    No `shutil.rmtree` against an existing project root.
    """
    if not archive.is_file():
        raise FileNotFoundError(f"Source archive not found: {archive}")

    archive_sha256 = sha256_file(archive)
    top_dir = _detect_zip_top_dir(archive)
    project_root = workspace / top_dir

    if project_root.is_dir() and verify_project_root(project_root) or False:
        pass

    if project_root.is_dir():
        if not force:
            print(
                f"Reusing existing source tree: {project_root}"
            )
            return project_root
        # If force=True we move the existing tree aside, never delete it.
        aside = workspace / f".{top_dir}.bak"
        n = 1
        while aside.exists():
            aside = workspace / f".{top_dir}.bak{n}"
            n += 1
        shutil.move(str(project_root), str(aside))
        print(
            f"force=True; moved existing tree aside: {aside}"
        )

    staging = workspace / (
        f".orthrus_pai_extract_staging_{archive_sha256[:12]}"
    )
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)

    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(staging)

    extracted_root = staging / top_dir
    if not extracted_root.is_dir():
        raise RuntimeError(
            f"Staging extraction missing top directory: {extracted_root}"
        )
    verify_project_root(extracted_root)

    shutil.move(str(extracted_root), str(project_root))
    shutil.rmtree(staging)
    print(f"Extracted source tree to: {project_root}")
    return project_root


def _candidate_preextracted_roots(
    workspace: Path,
    explicit,  # Optional[Path]
):
    """Bounded discovery of pre-extracted project root candidates.

    Order:
      1. Caller-supplied PREEXTRACTED_PROJECT_ROOT (exact-commit path).
      2. WORKSPACE_ROOT / 'orthrus-{COMMIT}' matching PRODUCTION_SOURCE_COMMIT.
      3. Direct child directory of (2).
      4. WORKSPACE_ROOT direct subdirs whose names start with
         'orthrus', plus one nested child directory of each.

    Never recurses into the workspace. /mnt/workspace/mstc_pids/artifacts
    is huge and must not be scanned.
    """
    candidates = []
    seen = set()

    def _add(label, p):
        try:
            resolved = p.resolve()
        except Exception:
            resolved = p
        if resolved in seen:
            return
        seen.add(resolved)
        candidates.append((label, p))

    if explicit is not None:
        _add("explicit PREEXTRACTED_PROJECT_ROOT", explicit)

    # Priority 2: exact-commit directory
    exact_commit_dir = workspace / f"orthrus-{PRODUCTION_SOURCE_COMMIT}"
    if exact_commit_dir.is_dir():
        _add("exact-commit directory", exact_commit_dir)
        for child in sorted(exact_commit_dir.iterdir()):
            if child.is_dir():
                _add("exact-commit child", child)

    # Legacy canonical path (low priority)
    canonical = workspace / "orthrus-fix-c8-loader-telemetry-persist"
    if canonical.is_dir():
        _add("legacy canonical (low priority)", canonical)
        for child in sorted(canonical.iterdir()):
            if child.is_dir():
                _add("legacy canonical child", child)

    try:
        for child in sorted(workspace.iterdir()):
            if not child.is_dir():
                continue
            if not child.name.startswith("orthrus"):
                continue
            if child.resolve() in seen:
                continue
            _add("workspace orthrus*", child)
            for grand in sorted(child.iterdir()):
                if grand.is_dir():
                    _add("workspace orthrus*/child", grand)
    except (FileNotFoundError, NotADirectoryError):
        pass

    return candidates


def find_preextracted_project_root(workspace: Path, explicit):
    """Locate a single valid pre-extracted project root.

    Returns:
        (root, [])          -- exactly one valid candidate.
        (None, [paths...])  -- multiple valid candidates; caller must
                               reject ambiguity rather than pick.
        (None, [])          -- no candidate passed verify_project_root.

    If the caller-supplied PREEXTRACTED_PROJECT_ROOT is itself valid,
    it wins over other auto-discovered candidates.
    """
    candidates = _candidate_preextracted_roots(workspace, explicit)
    valid = []
    for _, p in candidates:
        if not p.is_dir():
            continue
        try:
            verify_project_root(p)
        except FileNotFoundError:
            continue
        valid.append(p)

    if not valid:
        return None, []
    if len(valid) == 1:
        return valid[0], []
    if explicit is not None:
        for p in valid:
            try:
                if p.resolve() == explicit.resolve():
                    return p, [q for q in valid if q != p]
            except Exception:
                if p == explicit:
                    return p, [q for q in valid if q != p]
    return None, valid


def resolve_project_root(
    *,
    archive: Path,
    workspace: Path,
    explicit_preextracted,
    prefer_preextracted: bool,
    extract_source: bool,
):
    """Decide between pre-extracted source tree and ZIP fallback.

    Returns:
        (project_root, source_preparation_mode, source_archive_sha256)

    Modes:
        - 'preextracted': using a verified pre-extracted source tree.
          SOURCE_ARCHIVE_SHA256 is reported as None; the original ZIP
          may not be present, and we never forge a hash.
        - 'archive': prepared by prepare_source_archive() from the ZIP.
          SOURCE_ARCHIVE_SHA256 is the real SHA-256 of the ZIP.
    """
    preextracted, ambiguous = find_preextracted_project_root(
        workspace=workspace,
        explicit=explicit_preextracted,
    )

    if prefer_preextracted and preextracted is not None:
        return preextracted, "preextracted", None

    if ambiguous:
        listing = "\n".join("  - {}".format(p) for p in ambiguous)
        raise RuntimeError(
            "Multiple valid pre-extracted project roots were found, "
            "but PREEXTRACTED_PROJECT_ROOT does not match any of "
            "them. Refusing to pick one arbitrarily. Candidates:\n"
            + listing + "\n"
            "Set PREEXTRACTED_PROJECT_ROOT to the path you intend to "
            "use, or set PREFER_PREEXTRACTED_SOURCE=False to fall "
            "back to the GitHub ZIP."
        )

    if not extract_source:
        raise RuntimeError(
            "No valid pre-extracted source tree was found and "
            "EXTRACT_SOURCE=False, so ZIP fallback is disabled."
        )

    if not archive.is_file():
        raise FileNotFoundError(
            "No valid pre-extracted source tree was found and "
            "source archive does not exist: {}".format(archive)
        )

    project_root = prepare_source_archive(
        archive, workspace, force=False,
    )
    return project_root, "archive", sha256_file(archive)


def verify_source_provenance(project_root: Path) -> None:
    """Verify production source provenance via file content inspection.

    This replaces the Colab Notebook cross-check. The Colab Notebook
    is NOT a reliable source identity marker because it may pin a
    different branch/commit. Production source identity is verified by:
      1. PROJECT_ROOT is the expected exact-commit directory.
      2. Required source files exist.
      3. src/detection/evaluation.py contains correct import:
         from mstc.calibration_runner import
      4. src/detection/evaluation.py does NOT contain wrong import:
         from .mstc.calibration_runner import
      5. src/detection/orthrus_gnn_testing.py contains _cleanup_checkpoint_cuda.
      6. PRODUCTION_SOURCE_COMMIT is recorded.

    If the source was downloaded from the exact GitHub commit URL,
    record that provenance.
    """
    print("=" * 60)
    print("Source Provenance Verification")
    print("=" * 60)
    print(f"PRODUCTION_SOURCE_COMMIT = {PRODUCTION_SOURCE_COMMIT}")
    print(f"REPOSITORY_REF           = {REPOSITORY_REF}")
    print(f"PROJECT_ROOT             = {project_root}")

    # 1. Check PROJECT_ROOT matches expected exact-commit directory pattern
    expected_dir_pattern = f"orthrus-{PRODUCTION_SOURCE_COMMIT}"
    if expected_dir_pattern not in str(project_root):
        print(
            f"WARNING: PROJECT_ROOT does not contain expected commit hash.\n"
            f"  Expected pattern: {expected_dir_pattern}\n"
            f"  Got: {project_root}"
        )

    # 2. Check required source files
    print("\n[1/4] Verifying required source files...")
    verify_project_root(project_root)
    print("      Required source files: PASS")

    # 3. Check src/detection/evaluation.py import
    print("\n[2/4] Verifying src/detection/evaluation.py imports...")
    eval_py = project_root / "src" / "detection" / "evaluation.py"
    if not eval_py.is_file():
        raise FileNotFoundError(
            f"Required file missing: {eval_py}"
        )
    eval_content = eval_py.read_text(encoding="utf-8")

    # Correct import pattern
    if "from mstc.calibration_runner import" not in eval_content:
        raise AssertionError(
            f"src/detection/evaluation.py missing correct import:\n"
            f"  Expected: from mstc.calibration_runner import ...\n"
            f"  File: {eval_py}"
        )
    print("      Correct import 'from mstc.calibration_runner import': FOUND")

    # Wrong import pattern (relative with dot)
    if "from .mstc.calibration_runner import" in eval_content:
        raise AssertionError(
            f"src/detection/evaluation.py contains WRONG import:\n"
            f"  Found: from .mstc.calibration_runner import\n"
            f"  Should be: from mstc.calibration_runner import\n"
            f"  File: {eval_py}"
        )
    print("      Wrong import 'from .mstc.calibration_runner import': NOT FOUND (good)")

    # 4. Check src/detection/orthrus_gnn_testing.py has _cleanup_checkpoint_cuda
    print("\n[3/4] Verifying src/detection/orthrus_gnn_testing.py...")
    testing_py = project_root / "src" / "detection" / "orthrus_gnn_testing.py"
    if not testing_py.is_file():
        raise FileNotFoundError(
            f"Required file missing: {testing_py}"
        )
    testing_content = testing_py.read_text(encoding="utf-8")

    if "_cleanup_checkpoint_cuda" not in testing_content:
        raise AssertionError(
            f"src/detection/orthrus_gnn_testing.py missing _cleanup_checkpoint_cuda.\n"
            f"  File: {testing_py}"
        )
    print("      _cleanup_checkpoint_cuda: FOUND")

    # 5. Check for GitHub commit URL in source (if downloaded from URL)
    print("\n[4/4] Checking source provenance...")
    if "github.com" in str(project_root).lower() or "archive" in str(project_root).lower():
        print(f"      Source path suggests GitHub archive provenance")

    print("\n" + "=" * 60)
    print("Source Provenance Verification: PASSED")
    print("=" * 60)


## 2. Source Verification and Ground Truth Preparation

This section verifies that the uploaded source archive matches the
expected production ref, then extracts it into a staging directory
before swapping it in. The previous Colab Notebook
(fixed under `fix/c8-loader-telemetry-persist`) is cross-checked by
string match. No `rm -rf` is used against existing source.

In [ ]:
import sys
PROJECT_ROOT, SOURCE_PREPARATION_MODE, SOURCE_ARCHIVE_SHA256 = resolve_project_root(
    archive=SOURCE_ARCHIVE,
    workspace=WORKSPACE_ROOT,
    explicit_preextracted=PREEXTRACTED_PROJECT_ROOT,
    prefer_preextracted=PREFER_PREEXTRACTED_SOURCE,
    extract_source=EXTRACT_SOURCE,
)
verify_project_root(PROJECT_ROOT)
verify_source_provenance(PROJECT_ROOT)

print("=" * 60)
print("PAI Source Resolution")
print("=" * 60)
print("Preparation mode: {}".format(SOURCE_PREPARATION_MODE))
print("Project root:")
print("  {}".format(PROJECT_ROOT))
print("Source archive:")
print("  {}".format(SOURCE_ARCHIVE))
print("Archive exists: {}".format(SOURCE_ARCHIVE.is_file()))
print("Archive SHA256: {}".format(SOURCE_ARCHIVE_SHA256))
print("Required source files: PASS")
print("Source provenance verification: PASS")
print("=" * 60)

if SOURCE_PREPARATION_MODE == "preextracted":
    print(
        "Source ZIP is not present.\n"
        "Archive SHA256 is unavailable.\n"
        "Using verified pre-extracted source tree."
    )

(ENVIRONMENT_DIR / "source_archive_sha256.txt").write_text(
    "{}\n{}\n".format(SOURCE_ARCHIVE, SOURCE_ARCHIVE_SHA256 or ""),
    encoding="utf-8",
)

# Record PRODUCTION_SOURCE_COMMIT in environment
(ENVIRONMENT_DIR / "production_source_commit.txt").write_text(
    "{}\n".format(PRODUCTION_SOURCE_COMMIT),
    encoding="utf-8",
)

# Optional .git verification (only if a .git directory exists).
if (PROJECT_ROOT / ".git").is_dir():
    import subprocess
    head = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
        check=False, capture_output=True, text=True,
    )
    if head.returncode == 0:
        print("Git commit inside PROJECT_ROOT: {}".format(head.stdout.strip()))
    else:
        print("PROJECT_ROOT has .git but HEAD could not be resolved.")
else:
    print(
        "Source tree has no .git directory; "
        "git_commit will remain null in the environment record."
    )

# Add src to sys.path.
SRC_ROOT = str(PROJECT_ROOT / "src")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)
print("sys.path updated: {}".format(SRC_ROOT))


Run the helpers. `SOURCE_KIND` is `"github_zip"`; the archive has no
`.git`, so `git_commit` will remain `null` in the recorded environment
(see Section 24 — that's the documented behaviour, not a forgery).

SOURCE_ARCHIVE_SHA256 is recorded for later cross-checks.

Ground Truth is a Git submodule inside the project. On PAI-DSW we
(A) reuse a fully-prepared tree, (B) fall back to a local ZIP, or
(C) `git clone` from `ProvenanceAnalytics/ground-truth` and
`checkout` the pinned commit. HEAD is never used in place of the pin.

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

GROUND_TRUTH_ROOT = (
    PROJECT_ROOT / "Ground_Truth" / "darpa"
)
GROUND_TRUTH_REPO = (
    "https://github.com/ProvenanceAnalytics/ground-truth.git"
)
GROUND_TRUTH_COMMIT = (
    "012f321f46137650496e639b0ad7e0a66db07a73"
)
GROUND_TRUTH_ARCHIVE = (
    WORKSPACE_ROOT /
    f"ground-truth-{GROUND_TRUTH_COMMIT}.zip"
)

# Recovery path: verified persistent Ground Truth from previous run.
# If new source GT is placeholder-only, try to link to this.
PERSISTENT_GROUND_TRUTH_ROOT = Path(
    "/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist/"
    "orthrus-fix-c8-loader-telemetry-persist/"
    "Ground_Truth/darpa"
)


def _gt_already_initialised(root: Path) -> bool:
    """Heuristic: .git present, or any non-trivial data file present.

    Submodule data is committed via tags/releases rather than living
    on a single HEAD; we treat a populated tree as ready.
    """
    if not root.is_dir():
        return False
    if (root / ".git").is_file() or (root / ".git").is_dir():
        return True
    return any(p.is_file() for p in root.rglob("*") if p.is_file())


def _gt_has_theia_content(root: Path) -> bool:
    """Check if Ground Truth has actual E3-THEIA/E5-THEIA content.

    GitHub ZIP placeholders have no real data. We check for actual
    THEIA directory structure (nested layout: Ground_Truth/darpa/darpa/).
    Returns True only when real THEIA content exists.
    """
    if not root.is_dir():
        return False
    # Nested layout: Ground_Truth/darpa/darpa/E3-THEIA
    nested_e3 = root / "darpa" / "E3-THEIA"
    nested_e5 = root / "darpa" / "E5-THEIA"
    # Legacy layout: Ground_Truth/darpa/E3-THEIA
    legacy_e3 = root / "E3-THEIA"
    legacy_e5 = root / "E5-THEIA"
    return (
        nested_e3.is_dir() or nested_e5.is_dir() or
        legacy_e3.is_dir() or legacy_e5.is_dir()
    )


def _extract_archive(archive: Path, dest: Path) -> None:
    if dest.exists():
        shutil.move(
            str(dest),
            str(dest.with_name(dest.name + ".bak")),
        )
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(dest)


def ensure_ground_truth(
    *,
    root: Path,
    repo: str,
    commit: str,
    archive: Path,
    allow_network: bool,
    network_fetcher,
    persistent_gt_root: Path,
) -> Path:
    """Reuse / unzip / clone the ground-truth submodule data.

    Priority:
      1. New source GT has real E3-THEIA/E5-THEIA content -> use as-is
      2. Persistent GT path exists with real content -> verify + link
      3. Archive available -> extract
      4. Network allowed -> clone exact commit
      5. Otherwise -> error

    Never silently use wrong-version Ground Truth.
    """
    if _gt_has_theia_content(root):
        print(f"Ground Truth verified (new source): {root}")
        return root

    if persistent_gt_root is not None and _gt_has_theia_content(persistent_gt_root):
        print(f"Ground Truth from persistent path: {persistent_gt_root}")
        if root.exists():
            if not any(root.iterdir()):
                shutil.rmtree(root)
                root.symlink_to(persistent_gt_root, target_is_directory=True)
                print(f"Linked {root} -> {persistent_gt_root}")
            else:
                if not _gt_has_theia_content(root):
                    raise RuntimeError(
                        f"Source GT at {root} exists but has no THEIA content."
                    )
        else:
            root.symlink_to(persistent_gt_root, target_is_directory=True)
            print(f"Linked {root} -> {persistent_gt_root}")
        return root

    if archive.is_file():
        print(f"Extracting Ground Truth archive: {archive}")
        _extract_archive(archive, root)
        if _gt_has_theia_content(root):
            print(f"Ground Truth from archive verified: {root}")
            return root
        raise RuntimeError(
            f"Archive {archive} extracted but no THEIA content found."
        )

    if not allow_network:
        raise RuntimeError(
            "Ground Truth is incomplete. Enable ALLOW_NETWORK_FETCH_GROUND_TRUTH "
            "or upload matching archive."
        )

    print(f"Cloning Ground Truth from {repo} at {commit}")
    network_fetcher(root, repo, commit)
    if not _gt_has_theia_content(root):
        raise RuntimeError(
            f"Ground Truth clone failed to populate {root} with THEIA content."
        )
    print(f"Ground Truth from network verified: {root}")
    return root


In [ ]:
import subprocess


def _git_clone_ground_truth(
    dest: Path, repo: str, commit: str,
) -> None:
    """Network fetch: clone then strictly checkout the pinned commit.

    We never use the remote default branch HEAD in place of `commit`.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    if (dest / ".git").is_dir() or (dest / ".git").is_file():
        # Reuse a previous clone.
        subprocess.run(
            ["git", "-C", str(dest), "fetch", "--all"],
            check=False,
        )
    else:
        subprocess.run(
            ["git", "clone", repo, str(dest)],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(dest), "checkout", commit],
        check=True,
    )


In [ ]:
ensure_ground_truth(
    root=GROUND_TRUTH_ROOT,
    repo=GROUND_TRUTH_REPO,
    commit=GROUND_TRUTH_COMMIT,
    archive=GROUND_TRUTH_ARCHIVE,
    allow_network=ALLOW_NETWORK_FETCH_GROUND_TRUTH,
    network_fetcher=_git_clone_ground_truth,
    persistent_gt_root=PERSISTENT_GROUND_TRUTH_ROOT,
)
print(f"Pinned Ground Truth commit: {GROUND_TRUTH_COMMIT}")
resolved_gt = GROUND_TRUTH_ROOT
if resolved_gt.is_symlink():
    resolved_gt = resolved_gt.resolve()
print(f"Ground Truth root: {GROUND_TRUTH_ROOT}")
print(f"Resolved Ground Truth root: {resolved_gt}")


## 3. Resource Preflight

Inspect the PAI-DSW runtime: Python, torch, CUDA, GPU, CPU RAM,
workspace disk. Do **not** upgrade/downgrade torch or CUDA; the
pipeline expects the installed versions to be honoured.

In [ ]:
import shutil
import subprocess
import sys
import psutil
import torch

print(f"Python:     {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print(f"torch:      {torch.__version__}")
print(f"torch CUDA: {torch.version.cuda}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)

try:
    import torch_geometric
    print(f"PyG:        {torch_geometric.__version__}")
except Exception as exc:
    print(f"PyG import failed: {exc}")

ram = psutil.virtual_memory()
print(
    f"CPU RAM:    total={ram.total/1024**3:.1f} GB, "
    f"available={ram.available/1024**3:.1f} GB"
)

for label, path in (
    ("workspace", WORKSPACE_ROOT),
    ("mstc_root", MSTC_ROOT),
    ("artifact_root", ARTIFACT_ROOT),
    ("data_root", DATA_ROOT),
):
    if path.exists():
        usage = shutil.disk_usage(path)
        print(
            f"{label}: total={usage.total/1024**3:.1f} GB, "
            f"free={usage.free/1024**3:.1f} GB"
        )
    else:
        print(f"{label}: not found ({path})")

if SOURCE_ARCHIVE.is_file():
    print(
        f"Source archive size: "
        f"{SOURCE_ARCHIVE.stat().st_size / 1024**3:.2f} GB "
        f"({SOURCE_ARCHIVE})"
    )
else:
    print(f"Source archive missing: {SOURCE_ARCHIVE}")

if GROUND_TRUTH_ROOT.exists():
    n_files = sum(
        1 for _ in GROUND_TRUTH_ROOT.rglob("*") if _.is_file()
    )
    print(f"Ground Truth files (recursive): {n_files}")
else:
    print(f"Ground Truth not yet present: {GROUND_TRUTH_ROOT}")

GPU_READY = torch.cuda.is_available()
print(f"GPU_READY: {GPU_READY}")


## 4. Install / Verify Python Dependencies

The pipeline already ran the heavy 13GB preprocessing in the original
Google Drive; the artifacts live in `/mnt/workspace/mstc_pids/artifacts`.
We do **not** install/upgrade `torch` itself — we only install
missing pure-Python dependencies and the optional PyG compiled
extensions that match the currently installed `torch` + CUDA.

In [ ]:
import subprocess
import sys


def _pip_show(name: str) -> bool:
    """Return True when `pip show` reports the package as installed."""
    return subprocess.run(
        [sys.executable, "-m", "pip", "show", name],
        capture_output=True, text=True,
    ).returncode == 0


if not INSTALL_DEPENDENCIES:
    print("INSTALL_DEPENDENCIES=False; skipping pip install.")
else:
    missing = [
        pkg for pkg in (
            "scikit-learn", "networkx", "xxhash", "graphviz", "psutil",
            "matplotlib", "wandb", "chardet", "nltk", "igraph",
            "cairocffi", "wget", "gensim", "pytz", "pandas", "yacs",
            "psycopg2-binary", "tqdm", "pyyaml", "torch_geometric",
        )
        if not _pip_show(pkg)
    ]
    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", *missing],
            check=True,
        )
    else:
        print("All tracked pure-Python packages already installed.")

    # Optional PyG compiled extensions — pinned to the current torch + CUDA.
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = (
        "cpu" if torch.version.cuda is None
        else "cu" + torch.version.cuda.replace(".", "")
    )
    wheel_index = (
        f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    )
    print(f"PyG wheel index: {wheel_index}")

    for pkg in ("pyg_lib", "torch_scatter", "torch_sparse"):
        try:
            __import__(pkg)
            print(f"{pkg}: already importable")
        except Exception:
            if not _pip_show(pkg):
                try:
                    subprocess.run(
                        [
                            sys.executable, "-m", "pip", "install",
                            "--quiet", pkg, "-f", wheel_index,
                        ],
                        check=True,
                    )
                    print(f"{pkg}: installed")
                except subprocess.CalledProcessError as exc:
                    raise RuntimeError(
                        f"Failed to install {pkg}. "
                        f"Python={sys.version.split()[0]}, "
                        f"torch={torch.__version__}, "
                        f"CUDA={torch.version.cuda}, "
                        f"wheel_index={wheel_index}, "
                        f"pip rc={exc.returncode}."
                    ) from exc
            else:
                print(f"{pkg}: pip-show reports installed but import failed; "
                      "investigate manually before proceeding.")


## 5. Import Smoke Test and Environment Record

Smoke-import the core stack and persist a pip-freeze snapshot and an
`environment/pai_environment.json` record describing the runtime.

In [ ]:
import importlib
import subprocess
import sys
import json
from datetime import datetime, timezone

for module_name in (
    "torch", "torch_geometric", "pandas", "yaml", "gensim", "nltk",
    "networkx", "psutil", "sklearn",
):
    try:
        module = importlib.import_module(module_name)
        ver = getattr(module, "__version__", "")
        print(f"import {module_name}: ok ({ver})")
    except Exception as exc:
        print(f"import {module_name}: FAILED ({exc})")

# Pip freeze for environment record.
freeze_path = ENVIRONMENT_DIR / "pai_pip_freeze.txt"
with freeze_path.open("w", encoding="utf-8") as fh:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=fh, check=True, text=True,
    )
print(f"Environment freeze: {freeze_path}")


In [ ]:
env_record = {
    "platform": "aliyun_pai_dsw",
    "workspace_root": str(WORKSPACE_ROOT),
    "project_root": str(PROJECT_ROOT),
    "mstc_root": str(MSTC_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
    "data_root": str(DATA_ROOT),
    "source_archive": str(SOURCE_ARCHIVE),
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "source_kind": SOURCE_KIND,
    "source_preparation_mode": SOURCE_PREPARATION_MODE,
    "repository_ref": REPOSITORY_REF,
    "expected_production_commit": EXPECTED_PRODUCTION_COMMIT,
    "ground_truth_commit": GROUND_TRUTH_COMMIT,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "torch_geometric_version": getattr(
        importlib.import_module("torch_geometric"), "__version__", None
    ),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "git_commit": None,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

env_path = ENVIRONMENT_DIR / "pai_environment.json"
with env_path.open("w", encoding="utf-8") as fh:
    json.dump(env_record, fh, indent=2, ensure_ascii=False)
print(f"Environment record: {env_path}")
print(f"platform=aliyun_pai_dsw; git_commit=null (GitHub ZIP)")


## 6. Resolve Configuration and Inspect Persistent Artifacts

Reuse the production helper semantics to inspect preprocessing
artifacts. When `build_graphs`, `metadata`, `embed_nodes`,
`embed_edges` are all complete, the pipeline can run in
`detection_only` mode — no PostgreSQL restore needed.

In [ ]:
from config import get_runtime_required_args, get_yml_cfg
from pipeline_stages import (
    check_preprocess_stage_complete,
    check_all_preprocess_stages_complete,
)

PREPROCESS_CONFIG = PROJECT_ROOT / "config" / "orthrus.yml"


def fresh_preprocess_cfg():
    args = get_runtime_required_args(args=[
        DATASET, "--config", str(PREPROCESS_CONFIG),
        "--artifact-root", str(ARTIFACT_ROOT), "--stages", "preprocess",
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}",
        "--skip-tracing",
    ])
    return get_yml_cfg(args)


def artifact_stats(path):
    """Count + size of visible files under `path` (recursive)."""
    path = Path(path)
    if not path.is_dir():
        return {"files": 0, "MB": 0.0, "GB": 0.0}
    files = [p for p in path.rglob("*") if p.is_file()]
    total = sum(p.stat().st_size for p in files)
    return {
        "files": len(files),
        "MB": total / 1024**2,
        "GB": total / 1024**3,
    }


def read_preprocess_status(cfg):
    status = {
        stage: check_preprocess_stage_complete(cfg, stage)
        for stage in (
            "build_graphs", "embed_nodes", "embed_edges", "metadata",
        )
    }
    status["artifacts_complete"] = (
        check_all_preprocess_stages_complete(cfg)
    )
    return status


def print_status(label, status):
    print(label)
    for stage in (
        "build_graphs", "embed_nodes", "embed_edges", "metadata",
    ):
        print(f"  {stage}: {'Y' if status[stage] else 'N'}")
    print(f"  artifacts_complete: {status['artifacts_complete']}")


cfg = fresh_preprocess_cfg()
status_before = read_preprocess_status(cfg)
print_status("Current:", status_before)

artifact_paths = {
    "build_graphs": cfg.graph_construction.build_graphs._graphs_dir,
    "word2vec": cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir,
    "edge_embeddings": cfg.edge_featurization.embed_edges._edge_embeds_dir,
    "metadata": cfg._metadata_dir,
    "whole_ARTIFACT_ROOT": ARTIFACT_ROOT,
}
for label, p in artifact_paths.items():
    stats = artifact_stats(p)
    print(
        f"{label}: files={stats['files']}, "
        f"MB={stats['MB']:.2f}, GB={stats['GB']:.3f}"
    )

build_graphs = status_before["build_graphs"]
metadata = status_before["metadata"]
embed_nodes = status_before["embed_nodes"]
embed_edges = status_before["embed_edges"]
ALL_COMPLETE = status_before["artifacts_complete"]

DATABASE_REQUIRED_FOR_RESUME = (
    not build_graphs or not metadata
)
PREPROCESS_REQUIRED = not ALL_COMPLETE

if ALL_COMPLETE:
    print("\nDecision: ALL COMPLETE.")
    print("  - No database restore needed.")
    print("  - No preprocessing needed.")
    print("  - Next: GPU Baseline Smoke Test.")
elif DATABASE_REQUIRED_FOR_RESUME:
    print("\nDecision: Database restore required.")
    print("  - build_graphs or metadata incomplete.")
    print("  - Configure ORTHRUS_DB_* env vars + RUN_DB_PREFLIGHT=True.")
elif not embed_nodes or not embed_edges:
    print("\nDecision: Restartable preprocessing (no DB restore).")
    print("  - build_graphs + metadata complete.")
    print("  - Set RUN_PREPROCESS=True to resume embed_nodes/embed_edges.")
else:
    print("\nDecision: Status unclear; manual inspection required.")


## 7. Optional Database Preflight

PAI-DSW never auto-installs PostgreSQL. If artifacts are complete
we have zero database dependencies. When build_graphs or metadata
is missing, the user is responsible for providing an external
PostgreSQL endpoint via the `ORTHRUS_DB_*` environment variables.

In [ ]:
if not RUN_DB_PREFLIGHT:
    print("RUN_DB_PREFLIGHT=False; PostgreSQL untouched.")
else:
    import os
    import psycopg2

    user = os.environ.get("ORTHRUS_DB_USER") or ORTHRUS_DB_USER
    host = os.environ.get("ORTHRUS_DB_HOST") or ORTHRUS_DB_HOST
    port = os.environ.get("ORTHRUS_DB_PORT") or ORTHRUS_DB_PORT
    password = (
        os.environ.get("ORTHRUS_DB_PASSWORD") or ORTHRUS_DB_PASSWORD
    )
    if not (user and host and port and password):
        raise RuntimeError(
            "RUN_DB_PREFLIGHT=True but one or more of "
            "ORTHRUS_DB_HOST / PORT / USER / PASSWORD is empty."
        )
    connection = psycopg2.connect(
        user=user, host=host, port=int(port),
        password=password, dbname="postgres",
    )
    cursor = connection.cursor()
    try:
        cursor.execute("SELECT 1;")
        print(f"ORTHRUS database preflight OK at {host}:{port}")
    finally:
        cursor.close()
        connection.close()


## 8. Optional Bounded-Memory Benchmark

Preprocessing memory benchmark. Same script as Colab, but PAI
runs in CPU mode. Default skipped.

In [ ]:
# Section 8 — Optional Bounded-Memory Benchmark
# This section is skipped by default (RUN_PREPROCESS_BENCHMARK=False).
# Recovery helpers (run_streamed_command, MSTC_RECOVERY_SUCCEEDED) are
# defined in the Recovery Helper Definitions cell before Section 11.


## 9. Restartable Preprocessing

Restartable preprocessing. Uses `--cpu` because preprocessing has
no GPU benefit. Streamed output, exit-code check, completion-marker
verification, persistent log.

## 10. Baseline Smoke Test + Loader Telemetry Verification

The paper defines a smoke test as 1 epoch on `baseline.yml`, with
tracing disabled and `--max-windows-per-split 2`. We additionally
verify C8 loader telemetry persistence on the smoke output.

In [ ]:
if not RUN_BASELINE_SMOKE:
    print("RUN_BASELINE_SMOKE=False; bounded smoke skipped.")
elif not GPU_READY:
    print("GPU not available; smoke must NOT run on CPU.")
else:
    SMOKE_CONFIG = write_smoke_config()
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_experiment.py"),
        "--dataset", DATASET,
        "--config", str(SMOKE_CONFIG),
        "--seed", str(SEEDS[0]),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "train,test,evaluate",
        "--max-windows-per-split", str(SMOKE_MAX_WINDOWS_PER_SPLIT),
    ]
    print("=" * 60)
    print("Baseline Smoke Test")
    print("=" * 60)
    print(f"Dataset: {DATASET}, Seed: {SEEDS[0]}, Epochs: 1")
    print(f"Max windows/split: {SMOKE_MAX_WINDOWS_PER_SPLIT}")
    print(f"Config:  {SMOKE_CONFIG}")
    print(f"Command: {' '.join(cmd)}")

    rc = run_streamed_command(
        cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_baseline_smoke_latest.log",
    )
    if rc != 0:
        raise RuntimeError(
            f"Baseline smoke failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_baseline_smoke_latest.log'}."
        )
    print("=" * 60)
    print("Baseline bounded smoke PASSED")

    # C8 loader-telemetry acceptance (runs only when smoke succeeded).
    assert_loader_telemetry_persistence(ARTIFACT_ROOT)


In [ ]:
# ----------------------------------------------------------------------
# Recovery Helper Definitions
#
# These helpers are REQUIRED for Section 11 Recovery and must NOT be
# placed inside Optional Benchmark / Smoke / Main Matrix sections.
# They are defined here, immediately before the Recovery Preflight, so
# that users running Section 11 Recovery always execute them.
#
# Pure helpers (defined here — single authoritative location):
#   - run_streamed_command
#   - compute_config_id
#
# Recovery state:
#   - RECOVERY_MAIN_MATRIX  (canonical default: False, set in Section 0)
#   - MSTC_RECOVERY_SUCCEEDED  (False by default, set True after MSTC recovery)
# ----------------------------------------------------------------------

import hashlib
import subprocess
from pathlib import Path

# ----------------------------------------------------------------------
# run_streamed_command — single, authoritative definition
#
# Usage: rc = run_streamed_command(cmd, cwd=project_root, log_path=log)
#
# - subprocess (no shell=True)
# - stdout+stderr merged, real-time to Notebook and persistent log
# - log parent dir auto-created; flush after every write
# - blocks until process terminates; returns the true return code
# - does NOT raise on non-zero; caller handles rc != 0
# ----------------------------------------------------------------------
def run_streamed_command(cmd, *, cwd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = log_path.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in iter(proc.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
    proc.wait()
    log_file.flush()
    log_file.close()
    return proc.returncode


# Old config absolute paths — must use these for config-id identity preservation
OLD_BASELINE_CONFIG = Path(
    "/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist/"
    "orthrus-fix-c8-loader-telemetry-persist/"
    "config/experiments/baseline.yml"
)
OLD_MSTC_CONFIG = Path(
    "/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist/"
    "orthrus-fix-c8-loader-telemetry-persist/"
    "config/experiments/mstc_full.yml"
)

# Canonical config identities (production source 08ee62a)
EXPECTED_MSTC_ID = "mstc_full-d4b51c90f228"
EXPECTED_BASELINE_ID = "baseline-09b87fef8444"

# Artifact layout paths (derived from FORMAL_ARTIFACTS set by Section 8)
DATASET = "THEIA_E3"
MODEL_VARIANT = "orthrus"
SEED = 0

FORMAL_ARTIFACTS = ARTIFACT_ROOT  # /mnt/workspace/mstc_pids/artifacts

MSTC_MATRIX_ROOT = FORMAL_ARTIFACTS / "matrix_artifacts" / EXPECTED_MSTC_ID
MSTC_RUN_ROOT = MSTC_MATRIX_ROOT / DATASET / "runs" / MODEL_VARIANT / f"seed_{SEED}"
BASELINE_MATRIX_ROOT = FORMAL_ARTIFACTS / "matrix_artifacts" / EXPECTED_BASELINE_ID
BASELINE_RUN_ROOT = BASELINE_MATRIX_ROOT / DATASET / "runs" / MODEL_VARIANT / f"seed_{SEED}"

def compute_config_id(config_path: Path) -> str:
    """Derive stable config identity from absolute path (no config content needed).""" 
    component_re = __import__("re").compile(r"[^A-Za-z0-9._-]+")
    safe_stem = component_re.sub("-", config_path.stem).strip(".-") or "config"
    digest = hashlib.sha256(str(config_path).encode("utf-8")).hexdigest()[:12]
    return f"{safe_stem}-{digest}"

# Re-affirm recovery defaults (redundant but explicit, ensures idempotency)
if "RECOVERY_MAIN_MATRIX" not in dir():
    RECOVERY_MAIN_MATRIX = False
if "MSTC_RECOVERY_SUCCEEDED" not in dir():
    MSTC_RECOVERY_SUCCEEDED = False

print("Recovery helpers loaded.")
print(f"  EXPECTED_MSTC_ID     = {EXPECTED_MSTC_ID}")
print(f"  EXPECTED_BASELINE_ID = {EXPECTED_BASELINE_ID}")
print(f"  RECOVERY_MAIN_MATRIX = {RECOVERY_MAIN_MATRIX}")
print(f"  OLD_BASELINE_CONFIG  = {OLD_BASELINE_CONFIG}")
print(f"  OLD_MSTC_CONFIG      = {OLD_MSTC_CONFIG}")


In [ ]:
import hashlib
import json as _json_module
import os
import shutil
from pathlib import Path

# Canonical matrix artifact layout
EXPECTED_MSTC_ID = "mstc_full-d4b51c90f228"
EXPECTED_BASELINE_ID = "baseline-09b87fef8444"

DATASET = "THEIA_E3"
MODEL_VARIANT = "orthrus"
SEED = 0

MSTC_MATRIX_ROOT = FORMAL_ARTIFACTS / "matrix_artifacts" / EXPECTED_MSTC_ID
MSTC_RUN_ROOT = MSTC_MATRIX_ROOT / DATASET / "runs" / MODEL_VARIANT / f"seed_{SEED}"
BASELINE_MATRIX_ROOT = FORMAL_ARTIFACTS / "matrix_artifacts" / EXPECTED_BASELINE_ID
BASELINE_RUN_ROOT = BASELINE_MATRIX_ROOT / DATASET / "runs" / MODEL_VARIANT / f"seed_{SEED}"

OLD_BASELINE_CONFIG = Path(
    "/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist/"
    "orthrus-fix-c8-loader-telemetry-persist/"
    "config/experiments/baseline.yml"
)
OLD_MSTC_CONFIG = Path(
    "/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist/"
    "orthrus-fix-c8-loader-telemetry-persist/"
    "config/experiments/mstc_full.yml"
)

print("=" * 60)
print("SECTION 11 RECOVERY PREFLIGHT")
print("=" * 60)

# Step 1: Verify production commit
print("[1/5] Verifying production commit...")
if EXPECTED_PRODUCTION_COMMIT != "08ee62ac86d2c7e70f2bf49a18fd888e9227c448":
    raise AssertionError(
        "Production commit mismatch! Expected 08ee62a but got "
        f"{EXPECTED_PRODUCTION_COMMIT}. Aborting recovery."
    )
print(f"  Production commit verified: {EXPECTED_PRODUCTION_COMMIT}")

# Step 2: Verify Ground Truth
print()
print("[2/5] Verifying Ground Truth...")
if not _gt_has_theia_content(GROUND_TRUTH_ROOT):
    raise AssertionError(
        f"Ground Truth at {GROUND_TRUTH_ROOT} has no THEIA content. "
        "Recovery cannot proceed."
    )
resolved_gt = GROUND_TRUTH_ROOT
if resolved_gt.is_symlink():
    resolved_gt = resolved_gt.resolve()
print(f"  Resolved Ground Truth root: {resolved_gt}")
print("  Ground Truth verified: PASS")

# Step 3: Set up source/artifacts symlink
print()
print("[3/5] Verifying artifacts symlink...")
SOURCE_ARTIFACTS_LINK = PROJECT_ROOT / "artifacts"

if SOURCE_ARTIFACTS_LINK.is_symlink():
    target = SOURCE_ARTIFACTS_LINK.resolve()
    if target == FORMAL_ARTIFACTS.resolve():
        print(f"  artifacts symlink already points to formal root: {target}")
    else:
        raise RuntimeError(
            f"artifacts symlink points to unexpected location: {target}. "
            f"Expected: {FORMAL_ARTIFACTS}"
        )
elif SOURCE_ARTIFACTS_LINK.is_dir():
    if not any(SOURCE_ARTIFACTS_LINK.iterdir()):
        print("  Stale empty artifacts dir, removing and linking...")
        shutil.rmtree(SOURCE_ARTIFACTS_LINK)
        SOURCE_ARTIFACTS_LINK.symlink_to(FORMAL_ARTIFACTS, target_is_directory=True)
        print(f"  Linked {SOURCE_ARTIFACTS_LINK} -> {FORMAL_ARTIFACTS}")
    else:
        if (SOURCE_ARTIFACTS_LINK / "graph_construction").is_dir():
            print("  Source artifacts already contains data, reusing as-is")
        else:
            raise RuntimeError(
                f"Source artifacts directory exists and is non-empty but lacks expected content. "
                f"Manual inspection required at {SOURCE_ARTIFACTS_LINK}"
            )
else:
    SOURCE_ARTIFACTS_LINK.symlink_to(FORMAL_ARTIFACTS, target_is_directory=True)
    print(f"  Created symlink {SOURCE_ARTIFACTS_LINK} -> {FORMAL_ARTIFACTS}")

if SOURCE_ARTIFACTS_LINK.is_symlink():
    target = SOURCE_ARTIFACTS_LINK.resolve()
    print(f"  Verified symlink -> {target}")
else:
    print("  Using source artifacts directory directly")

# Step 4: Validate old config identity (MUST exist — no fallback)
print()
print("[4/5] Validating old config identity...")

if not OLD_BASELINE_CONFIG.is_file():
    raise FileNotFoundError(
        f"OLD baseline config not found: {OLD_BASELINE_CONFIG}"
        "  Recovery requires the original config path for identity preservation."
    )
baseline_id = compute_config_id(OLD_BASELINE_CONFIG)
if baseline_id != EXPECTED_BASELINE_ID:
    raise AssertionError(
        f"Baseline config-id mismatch! Expected {EXPECTED_BASELINE_ID}, got {baseline_id}. "
        "Recovery cannot proceed."
    )
print(f"  Baseline config-id: {baseline_id} == {EXPECTED_BASELINE_ID}: VERIFIED")

if not OLD_MSTC_CONFIG.is_file():
    raise FileNotFoundError(
        f"OLD MSTC config not found: {OLD_MSTC_CONFIG}"
        "  Recovery requires the original config path for identity preservation."
    )
mstc_id = compute_config_id(OLD_MSTC_CONFIG)
if mstc_id != EXPECTED_MSTC_ID:
    raise AssertionError(
        f"MSTC config-id mismatch! Expected {EXPECTED_MSTC_ID}, got {mstc_id}. "
        "Recovery cannot proceed."
    )
print(f"  MSTC config-id: {mstc_id} == {EXPECTED_MSTC_ID}: VERIFIED")

# Step 5: Verify MSTC evaluate-only prerequisites
print()
print("[5/5] Verifying MSTC evaluate-only prerequisites...")

mstc_ckpt_missing = []
for e in range(1, 7):
    ckpt = MSTC_RUN_ROOT / "checkpoints" / f"model_epoch_{e}" / "checkpoint.pt"
    status = "OK" if ckpt.is_file() else "MISSING"
    print(f"  MSTC checkpoint model_epoch_{e}: {status}")
    if not ckpt.is_file():
        mstc_ckpt_missing.append(e)
if mstc_ckpt_missing:
    raise RuntimeError(
        f"MSTC checkpoints MISSING for epochs: {mstc_ckpt_missing}. "
        "Cannot proceed to MSTC evaluate-only."
    )

mstc_val_missing = []
for e in range(1, 7):
    val_dir = MSTC_RUN_ROOT / "edge_scores" / "val" / f"model_epoch_{e}"
    csvs = list(val_dir.glob("*.csv")) if val_dir.is_dir() else []
    status = f"OK ({len(csvs)} CSVs)" if csvs else "MISSING/EMPTY"
    print(f"  MSTC val edge_scores model_epoch_{e}: {status}")
    if not csvs:
        mstc_val_missing.append(e)
if mstc_val_missing:
    raise RuntimeError(
        f"MSTC val edge_scores MISSING for epochs: {mstc_val_missing}. "
        "evaluation.py requires val edge_scores. Cannot proceed."
    )

mstc_test_missing = []
for e in range(1, 7):
    test_dir = MSTC_RUN_ROOT / "edge_scores" / "test" / f"model_epoch_{e}"
    csvs = list(test_dir.glob("*.csv")) if test_dir.is_dir() else []
    status = f"OK ({len(csvs)} CSVs)" if csvs else "MISSING/EMPTY"
    print(f"  MSTC test edge_scores model_epoch_{e}: {status}")
    if not csvs:
        mstc_test_missing.append(e)
if mstc_test_missing:
    raise RuntimeError(
        f"MSTC test edge_scores MISSING for epochs: {mstc_test_missing}. "
        "evaluation.py requires test edge_scores. Cannot proceed."
    )

# Verify Baseline checkpoints for downstream recovery
print()
print("[+] Verifying Baseline checkpoints (for test,evaluate)...")
baseline_ckpt_missing = []
for e in range(1, 7):
    ckpt = BASELINE_RUN_ROOT / "checkpoints" / f"model_epoch_{e}" / "checkpoint.pt"
    status = "OK" if ckpt.is_file() else "MISSING"
    print(f"  Baseline checkpoint model_epoch_{e}: {status}")
    if not ckpt.is_file():
        baseline_ckpt_missing.append(e)
if baseline_ckpt_missing:
    raise RuntimeError(
        f"Baseline checkpoints MISSING for epochs: {baseline_ckpt_missing}. "
        "Baseline test,evaluate requires all checkpoints. Cannot proceed."
    )

# Print run_status markers (read-only)
print()
print("[+] Checking run_status markers (read-only)...")
for label, config_id in [
    ("MSTC",    EXPECTED_MSTC_ID),
    ("Baseline", EXPECTED_BASELINE_ID),
]:
    status_file = (
        ARTIFACT_ROOT / "results" / "run_status" / DATASET /
        config_id / f"seed_{SEED}" / "run_status.json"
    )
    if status_file.is_file():
        status_data = _json_module.loads(status_file.read_text(encoding="utf-8"))
        print(f"  {label} run_status: {status_data.get('status')!r}")
    else:
        print(f"  {label} run_status: NOT FOUND")

print()
print("=" * 60)
print("SECTION 11 RECOVERY PREFLIGHT: ALL PASS")
print("Safe to proceed to MSTC Evaluate-Only recovery.")
print("=" * 60)


---

## Section 11 Recovery — MSTC Evaluate-Only

**Run this cell ONLY for MSTC recovery** after pre-flight checks pass.

MSTC evaluation was incomplete. Only `--stages evaluate` is allowed.
No `--force`, no `train`, no `test`.

Success criteria:
- `total_runs=1, completed=1, skipped=0, failed=0`
- `run_status.json` shows `status = "completed"`


In [ ]:
import json as _json_module
if not RECOVERY_MAIN_MATRIX:
    print("RECOVERY_MAIN_MATRIX=False; MSTC recovery skipped.")
elif not GPU_READY:
    print("GPU not available; recovery must NOT run on CPU.")
else:
    # Use OLD config path to preserve identity: mstc_full-d4b51c90f228
    MSTC_RECOVERY_CONFIG = OLD_MSTC_CONFIG

    print(f"Using old config path to preserve identity: {MSTC_RECOVERY_CONFIG}")

    # Verify config produces correct identity
    computed_id = compute_config_id(MSTC_RECOVERY_CONFIG)
    print(f"MSTC config-id: {computed_id}")
    assert computed_id == EXPECTED_MSTC_ID, (
        f"MSTC config-id mismatch: {computed_id} != {EXPECTED_MSTC_ID}"
    )

    mstc_recovery_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_matrix.py"),
        "--datasets", DATASET,
        "--configs", str(MSTC_RECOVERY_CONFIG),
        "--seeds", "0",
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "evaluate",  # ONLY evaluate, no train, no test
    ]

    # Use recovery-specific log name
    recovery_log = (
        ENVIRONMENT_DIR /
        f"mstc_evaluate_recovery_{EXPECTED_PRODUCTION_COMMIT[:12]}.log"
    )

    print("=" * 60)
    print("MSTC Recovery — Evaluate Only")
    print("=" * 60)
    print(f"Config: {MSTC_RECOVERY_CONFIG}")
    print(f"Config-id: {computed_id}")
    print(f"Stages: evaluate ONLY (no train, no test)")
    print(f"Command: {' '.join(mstc_recovery_cmd)}")
    print(f"Log: {recovery_log}")

    rc = run_streamed_command(
        mstc_recovery_cmd,
        cwd=PROJECT_ROOT,
        log_path=recovery_log,
    )

    if rc != 0:
        raise RuntimeError(
            f"MSTC recovery failed (rc={rc}). See {recovery_log}"
        )

    # Verify run_status.json and gate MSTC_RECOVERY_SUCCEEDED
    status_file = (
        ARTIFACT_ROOT / "results" / "run_status" / DATASET /
        EXPECTED_MSTC_ID / "seed_0" / "run_status.json"
    )
    print(f"Checking run status: {status_file}")
    if status_file.is_file():
        status_data = _json_module.loads(status_file.read_text(encoding="utf-8"))
        print(f"Run status: {status_data.get('status')}")
        if status_data.get("status") == "completed":
            print("MSTC Recovery SUCCESS!")
            MSTC_RECOVERY_SUCCEEDED = True
        else:
            raise RuntimeError(
                f"MSTC recovery status is {status_data.get('status')!r}, expected 'completed'."
            )
    else:
        raise RuntimeError(f"Status file not found at {status_file}. Cannot verify MSTC recovery.")

    print("=" * 60)
    print("MSTC Recovery Complete — Run Baseline recovery next if needed")
    print("=" * 60)


---

## Section 11 Recovery — Baseline Test + Evaluate

**Run this cell ONLY after MSTC recovery succeeds.**

Baseline `model_epoch_2 test` failed. Only `--stages test,evaluate` is allowed.
No `--force`, no `train`.

Success criteria:
- `total_runs=1, completed=1, skipped=0, failed=0`
- `run_status.json` shows `status = "completed"`


In [ ]:
import json as _json_module
if not RECOVERY_MAIN_MATRIX:
    print("RECOVERY_MAIN_MATRIX=False; Baseline recovery skipped.")
elif not GPU_READY:
    print("GPU not available; recovery must NOT run on CPU.")
elif not MSTC_RECOVERY_SUCCEEDED:
    raise RuntimeError(
        "MSTC recovery has not succeeded. "
        "Run MSTC Evaluate-Only recovery first, then retry Baseline recovery."
    )
else:
    # Use OLD config path to preserve identity: baseline-09b87fef8444
    BASELINE_RECOVERY_CONFIG = OLD_BASELINE_CONFIG

    print(f"Using old config path to preserve identity: {BASELINE_RECOVERY_CONFIG}")

    # Verify config produces correct identity
    computed_id = compute_config_id(BASELINE_RECOVERY_CONFIG)
    print(f"Baseline config-id: {computed_id}")
    assert computed_id == EXPECTED_BASELINE_ID, (
        f"Baseline config-id mismatch: {computed_id} != {EXPECTED_BASELINE_ID}"
    )

    baseline_recovery_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_matrix.py"),
        "--datasets", DATASET,
        "--configs", str(BASELINE_RECOVERY_CONFIG),
        "--seeds", "0",
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "test,evaluate",  # test + evaluate, no train
    ]

    # Use recovery-specific log name
    recovery_log = (
        ENVIRONMENT_DIR /
        f"baseline_test_evaluate_recovery_{EXPECTED_PRODUCTION_COMMIT[:12]}.log"
    )

    # Preserve old failure log
    OLD_FAILURE_LOG = ARTIFACT_ROOT / "environment" / "pai_main_matrix_failed_20260817.log"
    if (ENVIRONMENT_DIR / "pai_main_matrix_latest.log").is_file():
        import shutil as _shutil
        _shutil.copy(
            ENVIRONMENT_DIR / "pai_main_matrix_latest.log",
            OLD_FAILURE_LOG
        )
        print(f"Preserved failure log to: {OLD_FAILURE_LOG}")

    print("=" * 60)
    print("Baseline Recovery — Test + Evaluate")
    print("=" * 60)
    print(f"Config: {BASELINE_RECOVERY_CONFIG}")
    print(f"Config-id: {computed_id}")
    print(f"Stages: test,evaluate (no train)")
    print(f"Command: {' '.join(baseline_recovery_cmd)}")
    print(f"Log: {recovery_log}")

    rc = run_streamed_command(
        baseline_recovery_cmd,
        cwd=PROJECT_ROOT,
        log_path=recovery_log,
    )

    if rc != 0:
        raise RuntimeError(
            f"Baseline recovery failed (rc={rc}). See {recovery_log}"
        )

    # Verify run_status.json
    status_file = (
        ARTIFACT_ROOT / "results" / "run_status" / DATASET /
        EXPECTED_BASELINE_ID / "seed_0" / "run_status.json"
    )
    print(f"Checking run status: {status_file}")
    if status_file.is_file():
        status_data = _json_module.loads(status_file.read_text(encoding="utf-8"))
        print(f"Run status: {status_data.get('status')}")
        if status_data.get("status") == "completed":
            print("Baseline Recovery SUCCESS!")
        else:
            raise RuntimeError(
                f"Baseline recovery status is {status_data.get('status')!r}, expected 'completed'."
            )
    else:
        raise RuntimeError(f"Status file not found at {status_file}. Cannot verify Baseline recovery.")

    print("=" * 60)
    print("Baseline Recovery Complete — All recoveries done!")
    print("=" * 60)


## 11. Main Model Matrix

Main model matrix: `baseline.yml` + `mstc_full.yml` on THEIA_E3
(seed 0). No `--max-windows-per-split`. `--force` is **only** added
when the user explicitly opts in via `FORCE_MAIN_MATRIX=True`.

Streamed output goes to `pai_main_matrix_latest.log`; the
`run_matrix.py` scheduler owns `completed/skipped/failed` and
stale-running recovery. We do not re-implement any of that here.

In [ ]:
import subprocess
import sys

if not RUN_MAIN_MATRIX:
    print("RUN_MAIN_MATRIX=False; main model matrix skipped.")
elif not GPU_READY:
    print("GPU not available; main matrix must NOT run on CPU.")
else:
    MAIN_CONFIGS = [
        PROJECT_ROOT / "config" / "experiments" / "baseline.yml",
        PROJECT_ROOT / "config" / "experiments" / "mstc_full.yml",
    ]
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_matrix.py"),
        "--datasets", DATASET,
        "--configs", ",".join(map(str, MAIN_CONFIGS)),
        "--seeds", ",".join(map(str, SEEDS)),
        "--artifact-root", str(ARTIFACT_ROOT),
    ]
    if FORCE_MAIN_MATRIX:
        cmd.append("--force")

    print("=" * 60)
    print("Main Model Matrix")
    print("=" * 60)
    print(f"Configs: {[c.name for c in MAIN_CONFIGS]}")
    print(f"Datasets: {DATASET}; Seeds: {SEEDS}")
    print(f"Command: {' '.join(cmd)}")
    print()
    print("Monitor processes (live):")
    try:
        subprocess.run(
            ["ps", "-eo", "pid,etime,%cpu,%mem,cmd"],
            check=False,
        )
    except Exception:
        pass
    print(f"run_status markers: {ARTIFACT_ROOT / 'results' / 'run_status'}")
    print()

    rc = run_streamed_command(
        cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_main_matrix_latest.log",
    )
    if rc != 0:
        raise RuntimeError(
            f"Main matrix failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_main_matrix_latest.log'}."
        )
    print("=" * 60)
    print("Main model matrix complete.")


## 12. Ablations and Specialized Experiments

All ablation/specialized groups from the Colab Notebook, routed
through `run_matrix.py`. Same config sets, no semantic changes.

In [ ]:
import sys

GROUPS = {
    "ablation": [
        "baseline.yml", "ablation_no_multiscale.yml",
        "ablation_no_gate.yml", "ablation_no_time.yml",
        "ablation_no_calibration.yml", "ablation_no_topk.yml",
        "mstc_full.yml",
    ],
    "multiscale": [
        "multiscale_recent20.yml", "multiscale_recent24.yml",
        "multiscale_single_window.yml", "multiscale_equal.yml",
        "multiscale_gate.yml",
    ],
    "time": [
        "time_type_only.yml", "time_time_only.yml", "time_joint.yml",
    ],
    "calibration": [
        "calibration_max.yml", "calibration_quantile.yml",
        "calibration_kmeans.yml", "calibration_global_p.yml",
        "calibration_relation.yml", "calibration_hierarchical.yml",
    ],
    "backbone": [
        "backbone_graphtransformer.yml",
        "backbone_graphsage_baseline.yml",
        "backbone_graphsage.yml", "backbone_mlp.yml",
    ],
    "dataset_view": [
        "host_only.yml", "host_network_structure.yml",
        "host_network_full.yml",
    ],
    "efficiency": [
        "baseline.yml", "efficiency_multiscale.yml",
        "efficiency_multiscale_time.yml", "mstc_full.yml",
    ],
}

if not RUN_ABLATIONS:
    print("RUN_ABLATIONS=False; ablations skipped.")
elif not GPU_READY:
    print("GPU not available; ablations must NOT run on CPU.")
else:
    if EXPERIMENT_GROUP not in GROUPS:
        raise ValueError(
            f"Unknown EXPERIMENT_GROUP {EXPERIMENT_GROUP!r}; "
            f"available: {sorted(GROUPS)}"
        )
    cfg_paths = [
        PROJECT_ROOT / "config" / "experiments" / name
        for name in GROUPS[EXPERIMENT_GROUP]
    ]
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_matrix.py"),
        "--datasets", DATASET,
        "--configs", ",".join(map(str, cfg_paths)),
        "--seeds", ",".join(map(str, SEEDS)),
        "--artifact-root", str(ARTIFACT_ROOT),
    ]
    if FORCE_ABLATIONS:
        cmd.append("--force")

    print("=" * 60)
    print(f"Ablations: {EXPERIMENT_GROUP}")
    print("=" * 60)
    print(f"Configs: {GROUPS[EXPERIMENT_GROUP]}")
    print(f"Command: {' '.join(cmd)}")
    rc = run_streamed_command(
        cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_ablations_latest.log",
    )
    if rc != 0:
        raise RuntimeError(
            f"Ablations failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_ablations_latest.log'}."
        )
    print("Ablation experiments complete.")


## 13. Checkpoint Resume

Manual checkpoint resume. The user supplies `CHECKPOINT`; we only
run when `RUN_MANUAL_RESUME=True`. Missing checkpoint → FileNotFoundError.

In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_MANUAL_RESUME:
    print("RUN_MANUAL_RESUME=False; no checkpoint loaded.")
elif not GPU_READY:
    print("GPU not available; resume must NOT run on CPU.")
else:
    if not CHECKPOINT or not Path(CHECKPOINT).exists():
        raise FileNotFoundError(
            f"Checkpoint does not exist: {CHECKPOINT}"
        )
    RESUME_CONFIG = PROJECT_ROOT / RESUME_CONFIG_REL
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_experiment.py"),
        "--dataset", DATASET,
        "--config", str(RESUME_CONFIG),
        "--seed", str(SEEDS[0]),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "train,test,evaluate",
        "--checkpoint", str(CHECKPOINT),
    ]
    print("=" * 60)
    print("Checkpoint Resume")
    print("=" * 60)
    print(f"Config:    {RESUME_CONFIG}")
    print(f"Checkpoint:{CHECKPOINT}")
    print(f"Command: {' '.join(cmd)}")
    rc = run_streamed_command(
        cmd, cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_resume_latest.log",
    )
    if rc != 0:
        raise RuntimeError(f"Resume failed (rc={rc}).")
    print("Resume complete.")


## 14. Result Collection and Export

Collect results + export tables. The two helpers are the same
production scripts used by the Colab Notebook; semantics preserved.

In [ ]:
import subprocess
import sys

if not RUN_COLLECT_EXPORT:
    print("RUN_COLLECT_EXPORT=False; result collection/export skipped.")
else:
    print("=" * 60)
    print("Result Collection and Export")
    print("=" * 60)
    for script_name in ("collect_results.py", "export_tables.py"):
        cmd = [
            sys.executable,
            str(PROJECT_ROOT / "src" / "experiments" / script_name),
            "--artifact-root", str(ARTIFACT_ROOT),
        ]
        rc = run_streamed_command(
            cmd, cwd=PROJECT_ROOT,
            log_path=(
                ENVIRONMENT_DIR /
                f"pai_{script_name.replace('.py', '')}_latest.log"
            ),
        )
        if rc != 0:
            raise RuntimeError(
                f"{script_name} failed (rc={rc})."
            )
    print("Result collection and export complete.")


## 15. Result Display

Display the standard result tables. Missing tables produce a
warning rather than an error.

In [ ]:
import pandas as pd
from pathlib import Path

if not RUN_DISPLAY_RESULTS:
    print("RUN_DISPLAY_RESULTS=False; display skipped.")
else:
    print("=" * 60)
    print("Result Display")
    print("=" * 60)
    RESULTS_ROOT = ARTIFACT_ROOT / "results"
    tables = [
        "all_runs.csv",
        "main_results.csv",
        "ablation_results.csv",
        "calibration_results.csv",
        "efficiency_results.csv",
    ]
    for name in tables:
        path = RESULTS_ROOT / name
        if not path.is_file():
            print(f"Warning: {name} missing")
            continue
        df = pd.read_csv(path)
        print(f"\n--- {name} ({len(df)} rows) ---")
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.head().to_string())
    print("=" * 60)
